In [1]:
from pathlib import Path
import tifffile as tiff
import nd2


In [3]:
ROOT_DIR = Path("C:/Users/cowboy/OneDrive/Documents/University of Alabama")
DATA_DIR = ROOT_DIR / "Microscopy_Resources"
Img = "C:/Users/cowboy/OneDrive/Documents/University of Alabama/Microscopy_Resources/first025.nd2"


In [7]:
import nd2
from pathlib import Path

file_path = Path(r"C:\Users\cowboy\OneDrive\Documents\Unviversity of Alabama\Microscopy_Resources\first025.nd2")

with nd2.ND2File(file_path) as f:
    img = f.asarray()
    
    print("Shape:", img.shape)
    print("Axes:", f.sizes)

Shape: (2, 2304, 2304)
Axes: {'C': 2, 'Y': 2304, 'X': 2304}


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List, Optional, Sequence, Tuple, Union
import re

import numpy as np
import tifffile as tiff

try:
    import nd2
except ImportError:  # pragma: no cover
    nd2 = None


PathLike = Union[str, Path]
ArrayLike = np.ndarray


@dataclass
class ImageData:
    """
    Container for microscopy image data and axis metadata.

    Attributes
    ----------
    data
        Image array.
    axes
        Axis string describing `data`, e.g. "TZCYX".
    source_path
        Optional source file path.
    """

    data: np.ndarray
    axes: str
    source_path: Optional[Path] = None


# -----------------------------------------------------------------------------
# Axis utilities
# -----------------------------------------------------------------------------

def _normalize_axes_string(axes: str) -> str:
    """Normalize an axes string to uppercase with no whitespace."""
    axes = axes.upper().replace(" ", "")
    return axes


def ensure_axes(
    arr: np.ndarray,
    axes: str,
    target_axes: str = "TZCYX",
) -> np.ndarray:
    """
    Reorder an array from `axes` into `target_axes`, adding singleton dimensions
    for missing axes as needed.

    Parameters
    ----------
    arr
        Input array.
    axes
        Axis string describing the input array.
    target_axes
        Desired output axis order.

    Returns
    -------
    np.ndarray
        Array in the requested axis order.

    Notes
    -----
    - Axes present in the input but not in the target raise an error.
    - Missing axes in the target are added as singleton dimensions.
    - The input must contain Y and X unless your target also omits them.
    """
    axes = _normalize_axes_string(axes)
    target_axes = _normalize_axes_string(target_axes)

    extra_axes = [ax for ax in axes if ax not in target_axes]
    if extra_axes:
        raise ValueError(
            f"Input axes {axes!r} contain unsupported axes not in target {target_axes!r}: {extra_axes}"
        )

    out = arr
    working_axes = axes

    for ax in target_axes:
        if ax not in working_axes:
            out = np.expand_dims(out, axis=0)
            working_axes = ax + working_axes

    axis_map = {ax: i for i, ax in enumerate(working_axes)}
    order = [axis_map[ax] for ax in target_axes]
    out = np.transpose(out, axes=order)
    return out


def squeeze_unrequested_axes(
    arr: np.ndarray,
    axes: str,
    keep_axes: str,
) -> Tuple[np.ndarray, str]:
    """
    Remove singleton axes not listed in `keep_axes`.

    Useful after processing if you want to save or return a smaller-dimensional
    view, while preserving explicit axis labels.
    """
    axes = _normalize_axes_string(axes)
    keep_axes = _normalize_axes_string(keep_axes)

    out = arr
    out_axes = list(axes)

    for i in reversed(range(len(out_axes))):
        ax = out_axes[i]
        if ax not in keep_axes:
            if out.shape[i] != 1:
                raise ValueError(
                    f"Cannot drop non-singleton axis {ax!r} of size {out.shape[i]}"
                )
            out = np.squeeze(out, axis=i)
            out_axes.pop(i)

    return out, "".join(out_axes)


# -----------------------------------------------------------------------------
# File loading
# -----------------------------------------------------------------------------

def _require_nd2() -> None:
    if nd2 is None:
        raise ImportError(
            "The 'nd2' package is required to read ND2 files. Install it with: pip install nd2"
        )


def read_tiff(path: PathLike) -> ImageData:
    """
    Read a TIFF file and return ImageData with best-effort axis discovery.

    Raises
    ------
    ValueError
        If the TIFF axis metadata cannot be determined.
    """
    path = Path(path)
    with tiff.TiffFile(path) as tif:
        data = tif.asarray()
        axes = None

        if tif.series:
            try:
                axes = tif.series[0].axes
            except Exception:
                axes = None

    if axes is None:
        raise ValueError(
            f"Could not determine axis order for TIFF file: {path}. "
            "Inspect the file manually and provide a custom loader if needed."
        )

    return ImageData(data=data, axes=_normalize_axes_string(axes), source_path=path)


def read_nd2(path: PathLike) -> ImageData:
    """
    Read an ND2 file and return ImageData.

    Notes
    -----
    The `nd2` library typically reports dimension keys such as T, Z, C, Y, X.
    """
    _require_nd2()
    path = Path(path)

    with nd2.ND2File(path) as f:
        data = f.asarray()
        axes = "".join(f.sizes.keys())

    return ImageData(data=data, axes=_normalize_axes_string(axes), source_path=path)


def read_microscopy(path: PathLike) -> ImageData:
    """Read either TIFF or ND2 into an ImageData object."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in {".tif", ".tiff"}:
        return read_tiff(path)
    if suffix == ".nd2":
        return read_nd2(path)

    raise ValueError(f"Unsupported file extension for microscopy file: {path}")


def load_as_axes(path: PathLike, target_axes: str = "TZCYX") -> ImageData:
    """
    Read a microscopy file and return it in a standardized axis order.
    """
    img = read_microscopy(path)
    data = ensure_axes(img.data, img.axes, target_axes=target_axes)
    return ImageData(data=data, axes=_normalize_axes_string(target_axes), source_path=img.source_path)


# -----------------------------------------------------------------------------
# TIFF writing
# -----------------------------------------------------------------------------

def write_tiff(
    path: PathLike,
    data: np.ndarray,
    axes: str,
    *,
    imagej: bool = True,
    bigtiff: bool = True,
    dtype: Optional[np.dtype] = None,
    compression: Optional[str] = None,
) -> Path:
    """
    Write an array to TIFF with explicit axis metadata.

    Parameters
    ----------
    path
        Output TIFF path.
    data
        Array to write.
    axes
        Axis string for the array.
    imagej
        Whether to write ImageJ-compatible metadata.
    bigtiff
        Whether to force BigTIFF support.
    dtype
        Optional dtype cast before writing.
    compression
        Optional tifffile compression argument.
    """
    path = Path(path)
    arr = data.astype(dtype, copy=False) if dtype is not None else data
    axes = _normalize_axes_string(axes)

    path.parent.mkdir(parents=True, exist_ok=True)

    tiff.imwrite(
        path,
        arr,
        imagej=imagej,
        bigtiff=bigtiff,
        compression=compression,
        metadata={"axes": axes} if imagej else None,
    )
    return path


def convert_file_to_tiff(
    input_path: PathLike,
    output_path: Optional[PathLike] = None,
    *,
    target_axes: str = "TZCYX",
    dtype: Optional[np.dtype] = None,
    imagej: bool = True,
    bigtiff: bool = True,
    compression: Optional[str] = None,
) -> Path:
    """
    Convert a single ND2 or TIFF input into a TIFF with standardized axes.
    """
    input_path = Path(input_path)
    if output_path is None:
        output_path = input_path.with_suffix(".tif")
    output_path = Path(output_path)

    img = load_as_axes(input_path, target_axes=target_axes)
    return write_tiff(
        output_path,
        img.data,
        img.axes,
        imagej=imagej,
        bigtiff=bigtiff,
        dtype=dtype,
        compression=compression,
    )


# -----------------------------------------------------------------------------
# Concatenation
# -----------------------------------------------------------------------------

def _validate_compatible_shapes(arrays: Sequence[np.ndarray], axes: str, concat_axis: str) -> None:
    """Validate that all arrays match on non-concatenated axes."""
    concat_axis = _normalize_axes_string(concat_axis)
    axes = _normalize_axes_string(axes)

    if len(concat_axis) != 1:
        raise ValueError("concat_axis must be a single axis character")
    if concat_axis not in axes:
        raise ValueError(f"concat_axis {concat_axis!r} not present in axes {axes!r}")

    axis_index = axes.index(concat_axis)
    ref = arrays[0].shape

    for i, arr in enumerate(arrays[1:], start=1):
        if len(arr.shape) != len(ref):
            raise ValueError(
                f"Array {i} has {len(arr.shape)} dims but expected {len(ref)} dims"
            )
        for dim, (a, b) in enumerate(zip(arr.shape, ref)):
            if dim == axis_index:
                continue
            if a != b:
                raise ValueError(
                    f"Shape mismatch at non-concatenated axis {axes[dim]!r}: got {a}, expected {b}"
                )


def concatenate_files(
    input_paths: Sequence[PathLike],
    output_path: PathLike,
    *,
    target_axes: str = "TZCYX",
    concat_axis: str = "T",
    dtype: Optional[np.dtype] = None,
    imagej: bool = True,
    bigtiff: bool = True,
    compression: Optional[str] = None,
    verbose: bool = True,
) -> Path:
    """
    Load multiple microscopy files, standardize axes, concatenate, and save.

    This is the main function for combining multiple ND2/TIFF blocks into a
    single time series or z-stack.
    """
    if not input_paths:
        raise ValueError("input_paths is empty")

    target_axes = _normalize_axes_string(target_axes)
    concat_axis = _normalize_axes_string(concat_axis)
    axis_index = target_axes.index(concat_axis)

    arrays: List[np.ndarray] = []
    for p in input_paths:
        img = load_as_axes(p, target_axes=target_axes)
        arrays.append(img.data)
        if verbose:
            print(f"Loaded {Path(p).name}: shape={img.data.shape}, axes={img.axes}, dtype={img.data.dtype}")

    _validate_compatible_shapes(arrays, target_axes, concat_axis)
    out = np.concatenate(arrays, axis=axis_index)

    if verbose:
        print(f"Concatenated shape: {out.shape} ({target_axes})")

    return write_tiff(
        output_path,
        out,
        target_axes,
        imagej=imagej,
        bigtiff=bigtiff,
        dtype=dtype,
        compression=compression,
    )


def concatenate_hyperstacks_in_time(
    input_paths: Sequence[PathLike],
    output_path: PathLike,
    *,
    target_axes: str = "TZCYX",
    dtype: Optional[np.dtype] = None,
    imagej: bool = True,
    bigtiff: bool = True,
    compression: Optional[str] = None,
    verbose: bool = True,
) -> Path:
    """
    Convenience wrapper to concatenate along T.
    """
    return concatenate_files(
        input_paths,
        output_path,
        target_axes=target_axes,
        concat_axis="T",
        dtype=dtype,
        imagej=imagej,
        bigtiff=bigtiff,
        compression=compression,
        verbose=verbose,
    )


# -----------------------------------------------------------------------------
# Splitting
# -----------------------------------------------------------------------------

def split_along_axis(
    input_path: PathLike,
    output_dir: PathLike,
    *,
    axis: str = "T",
    target_axes: str = "TZCYX",
    prefix: Optional[str] = None,
    drop_split_axis: bool = False,
    dtype: Optional[np.dtype] = None,
    imagej: bool = True,
    bigtiff: bool = True,
    compression: Optional[str] = None,
    verbose: bool = True,
) -> List[Path]:
    """
    Split a microscopy file into separate TIFFs along a specified axis.

    Examples
    --------
    - Split a time series into one file per timepoint.
    - Split a z-stack into one file per z-plane.
    """
    img = load_as_axes(input_path, target_axes=target_axes)
    data = img.data
    axes = img.axes

    axis = _normalize_axes_string(axis)
    if len(axis) != 1 or axis not in axes:
        raise ValueError(f"Requested split axis {axis!r} is not present in axes {axes!r}")

    axis_index = axes.index(axis)
    n = data.shape[axis_index]

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    src_name = Path(input_path).stem
    prefix = prefix or src_name

    written: List[Path] = []
    for i in range(n):
        slicer = [slice(None)] * data.ndim
        slicer[axis_index] = slice(i, i + 1)
        chunk = data[tuple(slicer)]
        chunk_axes = axes

        if drop_split_axis:
            chunk, chunk_axes = squeeze_unrequested_axes(chunk, chunk_axes, keep_axes=chunk_axes.replace(axis, ""))

        out_path = output_dir / f"{prefix}_{axis.lower()}{i:04d}.tif"
        write_tiff(
            out_path,
            chunk,
            chunk_axes,
            imagej=imagej,
            bigtiff=bigtiff,
            dtype=dtype,
            compression=compression,
        )
        written.append(out_path)

        if verbose:
            print(f"Wrote {out_path.name}: shape={chunk.shape}, axes={chunk_axes}")

    return written


def split_hyperstack_by_time(
    input_path: PathLike,
    output_dir: PathLike,
    *,
    target_axes: str = "TZCYX",
    prefix: Optional[str] = None,
    drop_time_axis: bool = False,
    dtype: Optional[np.dtype] = None,
    imagej: bool = True,
    bigtiff: bool = True,
    compression: Optional[str] = None,
    verbose: bool = True,
) -> List[Path]:
    """Convenience wrapper to split a file along T."""
    return split_along_axis(
        input_path,
        output_dir,
        axis="T",
        target_axes=target_axes,
        prefix=prefix,
        drop_split_axis=drop_time_axis,
        dtype=dtype,
        imagej=imagej,
        bigtiff=bigtiff,
        compression=compression,
        verbose=verbose,
    )


# -----------------------------------------------------------------------------
# Deinterleaving / channel extraction
# -----------------------------------------------------------------------------

def extract_channels(
    input_path: PathLike,
    output_dir: PathLike,
    *,
    target_axes: str = "TZCYX",
    channel_names: Optional[Sequence[str]] = None,
    drop_channel_axis: bool = True,
    dtype: Optional[np.dtype] = None,
    imagej: bool = True,
    bigtiff: bool = True,
    compression: Optional[str] = None,
    verbose: bool = True,
) -> List[Path]:
    """
    Save one TIFF per channel from a multi-channel microscopy file.
    """
    img = load_as_axes(input_path, target_axes=target_axes)
    data = img.data
    axes = img.axes

    if "C" not in axes:
        raise ValueError(f"No channel axis present in axes {axes!r}")

    c_index = axes.index("C")
    n_channels = data.shape[c_index]

    if channel_names is None:
        channel_names = [f"channel_{i}" for i in range(n_channels)]
    if len(channel_names) != n_channels:
        raise ValueError(
            f"channel_names has length {len(channel_names)} but data has {n_channels} channels"
        )

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    stem = Path(input_path).stem
    written: List[Path] = []

    for c in range(n_channels):
        slicer = [slice(None)] * data.ndim
        slicer[c_index] = slice(c, c + 1)
        ch = data[tuple(slicer)]
        ch_axes = axes

        if drop_channel_axis:
            ch, ch_axes = squeeze_unrequested_axes(ch, ch_axes, keep_axes=ch_axes.replace("C", ""))

        safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", channel_names[c]).strip("_") or f"channel_{c}"
        out_path = output_dir / f"{stem}_{safe_name}.tif"

        write_tiff(
            out_path,
            ch,
            ch_axes,
            imagej=imagej,
            bigtiff=bigtiff,
            dtype=dtype,
            compression=compression,
        )
        written.append(out_path)

        if verbose:
            print(f"Wrote {out_path.name}: shape={ch.shape}, axes={ch_axes}")

    return written


def deinterleave_channels(
    input_path: PathLike,
    output_dir: PathLike,
    *,
    target_axes: str = "TZCYX",
    channel_names: Optional[Sequence[str]] = None,
    drop_channel_axis: bool = True,
    dtype: Optional[np.dtype] = None,
    imagej: bool = True,
    bigtiff: bool = True,
    compression: Optional[str] = None,
    verbose: bool = True,
) -> List[Path]:
    """
    Alias for extract_channels, using wording that matches microscopy workflows.
    """
    return extract_channels(
        input_path,
        output_dir,
        target_axes=target_axes,
        channel_names=channel_names,
        drop_channel_axis=drop_channel_axis,
        dtype=dtype,
        imagej=imagej,
        bigtiff=bigtiff,
        compression=compression,
        verbose=verbose,
    )


# -----------------------------------------------------------------------------
# Batch processing
# -----------------------------------------------------------------------------

def _natural_key(path: Path) -> List[Union[int, str]]:
    """Natural-sort helper for filenames containing integers."""
    parts = re.split(r"(\d+)", path.name.lower())
    key: List[Union[int, str]] = []
    for part in parts:
        if part.isdigit():
            key.append(int(part))
        else:
            key.append(part)
    return key


def find_microscopy_files(
    input_dir: PathLike,
    *,
    recursive: bool = False,
    suffixes: Sequence[str] = (".nd2", ".tif", ".tiff"),
) -> List[Path]:
    """Find microscopy files in a folder."""
    input_dir = Path(input_dir)
    suffixes = tuple(s.lower() for s in suffixes)

    if recursive:
        files = [p for p in input_dir.rglob("*") if p.is_file() and p.suffix.lower() in suffixes]
    else:
        files = [p for p in input_dir.iterdir() if p.is_file() and p.suffix.lower() in suffixes]

    return sorted(files, key=_natural_key)


def batch_convert_to_tiff(
    input_dir: PathLike,
    output_dir: PathLike,
    *,
    recursive: bool = False,
    suffixes: Sequence[str] = (".nd2", ".tif", ".tiff"),
    target_axes: str = "TZCYX",
    dtype: Optional[np.dtype] = None,
    imagej: bool = True,
    bigtiff: bool = True,
    compression: Optional[str] = None,
    overwrite: bool = False,
    verbose: bool = True,
) -> List[Path]:
    """
    Convert all supported microscopy files in a folder into standardized TIFFs.
    """
    files = find_microscopy_files(input_dir, recursive=recursive, suffixes=suffixes)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    written: List[Path] = []
    for src in files:
        dst = output_dir / f"{src.stem}.tif"
        if dst.exists() and not overwrite:
            if verbose:
                print(f"Skipping existing file: {dst.name}")
            written.append(dst)
            continue

        if verbose:
            print(f"Converting {src.name} -> {dst.name}")

        out_path = convert_file_to_tiff(
            src,
            dst,
            target_axes=target_axes,
            dtype=dtype,
            imagej=imagej,
            bigtiff=bigtiff,
            compression=compression,
        )
        written.append(out_path)

    return written


def batch_concatenate_matching_files(
    input_dir: PathLike,
    output_dir: PathLike,
    *,
    group_pattern: str,
    recursive: bool = False,
    suffixes: Sequence[str] = (".nd2", ".tif", ".tiff"),
    target_axes: str = "TZCYX",
    concat_axis: str = "T",
    dtype: Optional[np.dtype] = None,
    imagej: bool = True,
    bigtiff: bool = True,
    compression: Optional[str] = None,
    verbose: bool = True,
) -> List[Path]:
    """
    Group files by a regex capture and concatenate each group.

    Parameters
    ----------
    group_pattern
        Regular expression with one capture group used as the output stem.

    Example
    -------
    If files are named like:
        sampleA_part1.nd2
        sampleA_part2.nd2
        sampleB_part1.tif

    then a group_pattern like r"^(.*)_part\d+" will produce groups:
        sampleA -> [sampleA_part1, sampleA_part2]
        sampleB -> [sampleB_part1]
    """
    files = find_microscopy_files(input_dir, recursive=recursive, suffixes=suffixes)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    rx = re.compile(group_pattern)
    groups: dict[str, List[Path]] = {}

    for f in files:
        m = rx.match(f.stem)
        if not m:
            continue
        key = m.group(1)
        groups.setdefault(key, []).append(f)

    written: List[Path] = []
    for key, group_files in sorted(groups.items()):
        group_files = sorted(group_files, key=_natural_key)
        out_path = output_dir / f"{key}.tif"

        if verbose:
            names = ", ".join(p.name for p in group_files)
            print(f"Concatenating group {key}: {names}")

        written.append(
            concatenate_files(
                group_files,
                out_path,
                target_axes=target_axes,
                concat_axis=concat_axis,
                dtype=dtype,
                imagej=imagej,
                bigtiff=bigtiff,
                compression=compression,
                verbose=verbose,
            )
        )

    return written


# -----------------------------------------------------------------------------
# Inspection helpers
# -----------------------------------------------------------------------------

def summarize_file(path: PathLike, *, target_axes: Optional[str] = None) -> dict:
    """
    Return a concise metadata summary for an ND2 or TIFF file.
    """
    img = read_microscopy(path)
    summary = {
        "path": str(path),
        "source_axes": img.axes,
        "source_shape": tuple(img.data.shape),
        "dtype": str(img.data.dtype),
    }

    if target_axes is not None:
        target_axes = _normalize_axes_string(target_axes)
        arr = ensure_axes(img.data, img.axes, target_axes)
        summary["target_axes"] = target_axes
        summary["target_shape"] = tuple(arr.shape)

    return summary


def print_summary(path: PathLike, *, target_axes: Optional[str] = None) -> None:
    """Print a readable summary for quick debugging."""
    summary = summarize_file(path, target_axes=target_axes)
    for k, v in summary.items():
        print(f"{k}: {v}")


# -----------------------------------------------------------------------------
# Example usage
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    # Example 1: inspect a file
    # print_summary(r"C:\path\to\first025.nd2", target_axes="TZCYX")

    # Example 2: convert one ND2 to TIFF
    # convert_file_to_tiff(
    #     r"C:\path\to\first025.nd2",
    #     r"C:\path\to\converted\first025.tif",
    #     target_axes="TZCYX",
    # )

    # Example 3: concatenate blocks in explicit order
    # concatenate_hyperstacks_in_time(
    #     [
    #         r"C:\path\to\block_01.nd2",
    #         r"C:\path\to\block_02.tif",
    #     ],
    #     r"C:\path\to\output\full_series.tif",
    # )

    # Example 4: split one file into one TIFF per timepoint
    # split_hyperstack_by_time(
    #     r"C:\path\to\full_series.tif",
    #     r"C:\path\to\split_timepoints",
    #     drop_time_axis=True,
    # )

    # Example 5: deinterleave channels
    # deinterleave_channels(
    #     r"C:\path\to\full_series.tif",
    #     r"C:\path\to\channels",
    #     channel_names=["H2B", "Droplet", "ER"],
    # )

    pass


In [ ]:
convert_file_to_tiff()
concatenate_hyperstacks_in_time()
split_hyperstack_by_time()
deinterleave_channels()
batch_convert_to_tiff()
print_summary()

ND2 to tiff

In [ ]:
from microscopy_io_module import convert_file_to_tiff

convert_file_to_tiff(
    r"C:\Users\cowboy\OneDrive\Documents\Unviversity of Alabama\Microscopy_Resources\first025.nd2",
    r"C:\Users\cowboy\OneDrive\Documents\Unviversity of Alabama\Microscopy_Resources\first025.tif",
    target_axes="TZCYX",
)